# Bases de datos y datasets — Ejemplos prácticos

**Módulo 1 — Introducción · Curso Analítica de Datos**

Este notebook acompaña las diapositivas [`1.3_Bases_de_datos_y_datasets.pdf`](1.3_Bases_de_datos_y_datasets.pdf) y muestra en código lo que allí se explica: **de dónde vienen los datos** y **en qué formatos podemos encontrarlos**. Trabajaremos con el [UCI Machine Learning Repository](https://archive.ics.uci.edu/), uno de los repositorios de datasets más usados en la comunidad de ciencia de datos.

## Contenido

1. **Dataset en formato CSV** (UCI — *Wine Quality*): descarga, exploración con Pandas y una visualización con Matplotlib.
2. **Dataset en formato Raw Text** (UCI — *Auto MPG*): descarga y *parsing* de texto plano sin la comodidad de comas, exploración y visualización.
3. **Base de datos SQL** (SQLite): cómo cargar datos en una base de datos relacional, consultarlos con SQL y visualizar el resultado.

> 💡 Las secciones 1 y 2 **requieren conexión a internet** (descargan los datos directamente del sitio de la UCI). La sección 3 no la necesita: usa SQLite, una base de datos que no requiere instalar ningún servidor.

In [ ]:
# Librerías que usaremos en todo el notebook
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Para que las tablas se vean bien y los gráficos tengan un tamaño cómodo por defecto
pd.set_option('display.max_columns', 20)
plt.rcParams['figure.figsize'] = (8, 5)

print('Librerías cargadas correctamente ✅')
print('pandas', pd.__version__)

---
## 1. Dataset en formato CSV — *Wine Quality* (UCI)

El [**Wine Quality Dataset**](https://archive.ics.uci.edu/dataset/186/wine+quality) contiene mediciones fisicoquímicas (acidez, azúcar residual, alcohol, pH, etc.) de más de 1500 vinos tintos portugueses, junto con una calificación de calidad (de 3 a 8) dada por catadores expertos. Es un dataset muy usado para practicar tanto análisis exploratorio como modelos de predicción.

Como vimos en las diapositivas, el formato **CSV (Comma-Separated Values)** representa cada fila como un registro y separa sus columnas con un delimitador — normalmente una coma, aunque también son comunes el punto y coma (`;`, como en este caso) o el tabulador (formato TSV).

In [ ]:
# URL del archivo CSV directamente en el repositorio de la UCI.
# OJO: este archivo en particular usa punto y coma (;) como separador, no coma —
# un buen recordatorio de que "CSV" es una familia de formatos, no un único estándar.
url_vinos = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'

try:
    vinos = pd.read_csv(url_vinos, sep=';')
    print(f'Dataset descargado correctamente: {vinos.shape[0]} filas x {vinos.shape[1]} columnas')
except Exception as error:
    # Si no hay conexión a internet o la UCI no responde, avisamos con un mensaje claro
    # en vez de dejar que el estudiante vea un traceback críptico.
    print('❌ No se pudo descargar el dataset. Verifica tu conexión a internet.')
    print('Detalle del error:', error)
    raise

In [ ]:
# Exploración inicial con Pandas: así se ve el dataset por dentro
vinos.head()

In [ ]:
# Información general: cuántas filas, columnas, tipos de dato y si hay nulos
vinos.info()

In [ ]:
# Resumen estadístico de todas las columnas numéricas: media, desviación,
# mínimo, máximo y cuartiles — muy útil como primer vistazo a la escala de cada variable
vinos.describe().round(2)

In [ ]:
# ¿Cuántos vinos hay de cada nivel de calidad? La mayoría de las calificaciones
# están concentradas en 5 y 6; hay muy pocos vinos calificados como excelentes (8) o malos (3).
vinos['quality'].value_counts().sort_index()

### Visualización: ¿el alcohol influye en la calidad percibida del vino?

Calculamos el contenido promedio de alcohol para cada nivel de calidad y lo graficamos como barras — una forma sencilla de detectar si existe una tendencia.

In [ ]:
# Promedio de alcohol agrupado por nivel de calidad (groupby + mean, ordenado por índice)
alcohol_por_calidad = vinos.groupby('quality')['alcohol'].mean().sort_index()

fig, ax = plt.subplots()

# Barras coloreadas con un degradado (colormap) según el nivel de calidad,
# para que la gráfica comunique visualmente "de menor a mayor calidad"
colores = plt.cm.viridis((alcohol_por_calidad.index - alcohol_por_calidad.index.min())
                          / (alcohol_por_calidad.index.max() - alcohol_por_calidad.index.min()))
barras = ax.bar(alcohol_por_calidad.index.astype(str), alcohol_por_calidad.values, color=colores)

# Etiqueta con el valor exacto encima de cada barra
for barra, valor in zip(barras, alcohol_por_calidad.values):
    ax.text(barra.get_x() + barra.get_width() / 2, valor + 0.05, f'{valor:.1f}%',
            ha='center', va='bottom', fontsize=9)

ax.set_title('Contenido promedio de alcohol según la calidad del vino')
ax.set_xlabel('Calidad (calificación de 3 a 8)')
ax.set_ylabel('Alcohol promedio (% vol.)')
ax.set_ylim(0, alcohol_por_calidad.max() + 1)
plt.tight_layout()
plt.show()

print('Tendencia: los vinos mejor calificados tienden a tener, en promedio, más contenido de alcohol.')

---
## 2. Dataset en formato Raw Text — *Auto MPG* (UCI)

El [**Auto MPG Dataset**](https://archive.ics.uci.edu/dataset/9/auto+mpg) describe el consumo de combustible (millas por galón) de más de 390 modelos de carro de los años 70 y 80, junto con características técnicas: cilindraje, potencia, peso, aceleración, año del modelo y país de origen.

A diferencia del CSV, este archivo viene en **texto plano sin delimitador fijo**: los valores están separados por una cantidad variable de espacios (para que, al abrirlo en un editor de texto, las columnas queden alineadas visualmente), los valores faltantes se marcan con `?` en vez de dejarse vacíos, y el nombre del carro viene entre comillas al final de cada línea, separado del resto por un tabulador. Este tipo de formato exige más trabajo de *parsing* que un CSV — justo lo que mencionan las diapositivas sobre la diferencia entre "Raw Text" y formatos con estructura explícita.

In [ ]:
# URL del archivo de texto plano en el repositorio de la UCI
url_autos = 'https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'

# El archivo no trae encabezado, así que definimos los nombres de columna manualmente
# (en el mismo orden documentado por la UCI). Omitimos a propósito el nombre del carro:
# viene precedido por un tabulador, y usamos ese tabulador como marca de "comentario"
# para que pandas descarte el resto de la línea y no tengamos que lidiar con las comillas.
columnas_autos = ['mpg', 'cilindros', 'desplazamiento', 'caballos_fuerza', 'peso',
                   'aceleracion', 'anio_modelo', 'origen']

try:
    autos = pd.read_csv(
        url_autos,
        names=columnas_autos,
        na_values='?',       # los valores faltantes están marcados con "?" en vez de vacíos
        comment='\t',        # descarta todo lo que viene después del tabulador (el nombre del carro)
        sep=' ',             # separador base: espacio
        skipinitialspace=True,  # ignora los espacios extra usados para alinear las columnas
    )
    print(f'Dataset descargado correctamente: {autos.shape[0]} filas x {autos.shape[1]} columnas')
except Exception as error:
    print('❌ No se pudo descargar el dataset. Verifica tu conexión a internet.')
    print('Detalle del error:', error)
    raise

In [ ]:
autos.head()

In [ ]:
# ¿Cuántos valores faltantes quedaron marcados como NaN? (deberían estar solo en 'caballos_fuerza')
autos.isna().sum()

In [ ]:
# Para graficar sin advertencias, quitamos las pocas filas con datos faltantes.
# 'cilindros' y 'anio_modelo' llegan como enteros; los pasamos a int para que se vean
# limpios en las gráficas (sin ".0").
autos_limpio = autos.dropna().copy()
autos_limpio['cilindros'] = autos_limpio['cilindros'].astype(int)
autos_limpio['anio_modelo'] = autos_limpio['anio_modelo'].astype(int)

autos_limpio.describe().round(1)

### Visualización: potencia vs. consumo, coloreado por número de cilindros

Un carro con más cilindros suele ser más potente mecánicamente, pero también más pesado y menos eficiente. Veamos si eso se refleja en los datos.

In [ ]:
fig, ax = plt.subplots()

# Scatter plot: caballos de fuerza vs mpg, coloreado según el número de cilindros
dispersión = ax.scatter(
    autos_limpio['caballos_fuerza'], autos_limpio['mpg'],
    c=autos_limpio['cilindros'], cmap='plasma', s=45, alpha=0.85, edgecolor='white',
)

barra_color = fig.colorbar(dispersión, ax=ax)
barra_color.set_label('Número de cilindros')
# Mostrar solo valores enteros en la barra de color (4, 6, 8...)
barra_color.locator = mticker.MaxNLocator(integer=True)
barra_color.update_ticks()

ax.set_title('Relación entre potencia y consumo de combustible')
ax.set_xlabel('Potencia (caballos de fuerza)')
ax.set_ylabel('Eficiencia (millas por galón)')
plt.tight_layout()
plt.show()

print('Tendencia: a mayor potencia (y más cilindros), menor eficiencia de combustible.')

---
## 3. Base de datos SQL — SQLite

Como vimos en las diapositivas, las **bases de datos relacionales** organizan la información en tablas conectadas y permiten consultarla con **SQL**, con mejor rendimiento y control que una hoja de cálculo cuando el volumen de datos crece.

Para este ejemplo usamos **SQLite**: un motor de base de datos SQL real, pero que no requiere instalar ni configurar ningún servidor — toda la base vive en un único archivo (o incluso en memoria). Viene incluido con Python (módulo `sqlite3`), así que este ejemplo funciona sin conexión a internet.

> Si más adelante trabajas con un motor "de verdad" (MySQL, PostgreSQL, SQL Server), el flujo es el mismo: cambia la forma de conectarse (por ejemplo con `sqlalchemy` + `pymysql`/`psycopg2`), pero `pd.read_sql_query()` funciona igual.

In [ ]:
import sqlite3

# Creamos (o abrimos) un archivo de base de datos SQLite en la carpeta actual.
conexion = sqlite3.connect('vinos.db')

# Cargamos el DataFrame de vinos (de la Sección 1) como una tabla SQL llamada 'vinos'.
# if_exists='replace': si ya existe la tabla de una ejecución anterior, la reemplaza.
vinos.to_sql('vinos', conexion, if_exists='replace', index=False)

print('Tabla "vinos" creada en la base de datos SQLite ✅')

In [ ]:
# Una consulta SQL típica: agrupar, contar y promediar — igual que el groupby de
# Pandas, pero expresado en SQL, tal como se haría contra cualquier base de datos real.
consulta_sql = '''
SELECT
    quality AS calidad,
    COUNT(*) AS num_vinos,
    ROUND(AVG(pH), 2) AS ph_promedio
FROM vinos
GROUP BY quality
ORDER BY quality;
'''

resumen_sql = pd.read_sql_query(consulta_sql, conexion)
resumen_sql

### Visualización: cantidad de vinos y pH promedio por nivel de calidad

Combinamos dos ejes en una misma gráfica: barras para la cantidad de vinos de cada calidad, y una línea para el pH promedio de ese grupo.

In [ ]:
fig, ax1 = plt.subplots()

# Eje izquierdo: barras con la cantidad de vinos por nivel de calidad
ax1.bar(resumen_sql['calidad'].astype(str), resumen_sql['num_vinos'], color='#4C72B0', alpha=0.85)
ax1.set_xlabel('Calidad (calificación de 3 a 8)')
ax1.set_ylabel('Cantidad de vinos', color='#4C72B0')
ax1.tick_params(axis='y', labelcolor='#4C72B0')

# Eje derecho: línea con el pH promedio de cada grupo, compartiendo el mismo eje X
ax2 = ax1.twinx()
ax2.plot(resumen_sql['calidad'].astype(str), resumen_sql['ph_promedio'],
          color='#C44E52', marker='o', linewidth=2)
ax2.set_ylabel('pH promedio', color='#C44E52')
ax2.tick_params(axis='y', labelcolor='#C44E52')

plt.title('Cantidad de vinos y pH promedio por nivel de calidad (consulta SQL)')
fig.tight_layout()
plt.show()

In [ ]:
# Buena práctica: cerrar la conexión cuando ya no vamos a hacer más consultas.
conexion.close()
print('Conexión a la base de datos cerrada ✅')

---
## Cierre

En este notebook vimos, en código, las ideas de las diapositivas [`1.3_Bases_de_datos_y_datasets.pdf`](1.3_Bases_de_datos_y_datasets.pdf):

- Cómo acceder y cargar datasets del **UCI Machine Learning Repository** en dos formatos distintos: **CSV** (con un delimitador explícito) y **Raw Text** (sin delimitador fijo, con marcadores propios para valores faltantes).
- Que un mismo dataset puede explorarse con **Pandas** y visualizarse con **Matplotlib** sin importar de qué formato vino, una vez está cargado en un DataFrame.
- Cómo cargar datos en una **base de datos SQL** (SQLite), consultarlos con SQL real, y traer el resultado de vuelta a Pandas para graficarlo.

### Recursos adicionales
- [UCI Machine Learning Repository](https://archive.ics.uci.edu/)
- [Documentación de `pandas.read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)
- [Documentación del módulo `sqlite3`](https://docs.python.org/3/library/sqlite3.html)